https://www.kaggle.com/datasets/nphantawee/pump-sensor-data?resource=download

https://medium.com/kx-systems/time-series-similarity-search-for-iot-sensor-failure-detection-6573de6c55e4

In [64]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import  f1_score, accuracy_score, precision_score, recall_score

import matplotlib

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
from sklearn.metrics import confusion_matrix
from scipy import stats

from transformations import normalizer

from model_test import model_test
from model_test import model_test_new

from datasampling import window_duplication_oversample
from datasampling import ohit_time_segments_oversample
from datasampling import knn_time_oversample
from datasampling import drsnn_oversample_df

from transformations import test_train_split

print(f"Pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")

Pandas: 2.2.3
Matplotlib: 3.9.3


In [65]:
df_sensor = pd.read_parquet('pump_sensor_data_clean.pqt')

sensor_metric_columns = [col for col in df_sensor.columns if col.startswith('metric')]

In [66]:
df_sensor = normalizer(df_sensor,sensor_metric_columns)
df_sensor = df_sensor.sort_values('timestamp').reset_index(drop=True)

df_sensor.to_parquet("pump_sensor_data_clean.pqt")

In [67]:
df_sensor_broken = df_sensor[df_sensor['failure'] == 1]
df_sensor_recovery = df_sensor[df_sensor['failure'] == 2]

In [68]:
df_metric_check = df_sensor.copy()
#analysis_by_metrics(df_metric_check,10)

## Data Transformations
To understand how the data is performing with the different models, it was transformed to binary classes. For this specific case, there were 3 machine statuses and the interim status was transformed into either a failure or normal status.

The dataset was also split into training and test sets with a clear time separation. This ensures, that the training and test data is not overlayed and clearly separated.

In [69]:
# Combine “RECOVERING” and “BROKEN” into a Single Class
df_sensor_notnormal = df_sensor.copy()
df_sensor_notnormal['failure'] = df_sensor_notnormal['failure'].replace({
    2: 1
})

# Merge “RECOVERING” with "NORMAL"
df_sensor_normal = df_sensor.copy()
df_sensor_normal['failure'] = df_sensor_normal['failure'].replace({
    2: 0
})

In [70]:
split_time = '2018-07-01'

In [71]:
# Split the data to test and train by the specific date
X_train_nn, X_test_nn, y_train_nn, y_test_nn = test_train_split(df_sensor_notnormal, split_time)
X_train_n, X_test_n, y_train_n, y_test_n = test_train_split(df_sensor_normal, split_time)

In [72]:
""""
Create new dataframes for the model testing using the different sampling techiques:
DRSNN: drsnn_oversample_df
Window Duplication: window_duplication_oversample

For the different sampling thechniques, there is a separate dataframe created, which will be tested by the different models
"""
df_to_sample = df_sensor_notnormal # define the dataframe to be used for the sampling process

df_sensor_drsnn = drsnn_oversample_df(df_to_sample)
df_sensor_drsnn = df_sensor_drsnn.sort_values(by="timestamp").reset_index(drop=True)
df_sensor_drsnn_broken = df_sensor_drsnn[df_sensor_drsnn['failure'] == 1]

df_sensor_window = window_duplication_oversample(df_to_sample)
df_sensor_window = df_sensor_window.sort_values(by="timestamp").reset_index(drop=True)
df_sensor_window_broken = df_sensor_window[df_sensor_window['failure'] == 1]


In [73]:
# Create test train dataset for the sampling functions
tscv = TimeSeriesSplit(n_splits=5)

X_train_drsnn, X_test_drsnn, y_train_drsnn, y_test_drsnn = test_train_split(df_sensor_drsnn, split_time)
X_train_w, X_test_w, y_train_w, y_test_w = test_train_split(df_sensor_window, split_time)

# Time series split of the sampling datasets
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_drsnn)):
    X_train_fold_drsnn, X_val_fold_drsnn = X_train_drsnn.iloc[train_idx], X_train_drsnn.iloc[val_idx]
    y_train_fold_drsnn, y_val_fold_drsnn = y_train_drsnn.iloc[train_idx], y_train_drsnn.iloc[val_idx]

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_w)):
    X_train_fold_w, X_val_fold_w = X_train_w.iloc[train_idx], X_train_w.iloc[val_idx]
    y_train_fold_w, y_val_fold_w = y_train_w.iloc[train_idx], y_train_w.iloc[val_idx]

### Final validation of the selected model / sampling
Based on the test results the following combinations are showing the highest performance
- DRSNN + GradientBoost
- Window Duplication + GradientBoost

In [74]:
df_sensor_drsnn_norm = drsnn_oversample_df(df_sensor_normal)
df_sensor_drsnn_norm = df_sensor_drsnn.sort_values(by="timestamp").reset_index(drop=True)

df_sensor_window_norm = window_duplication_oversample(df_sensor_normal)
df_sensor_window_norm = df_sensor_window.sort_values(by="timestamp").reset_index(drop=True)

In [75]:
X_train_drsnn, X_test_drsnn, y_train_drsnn, y_test_drsnn = test_train_split(df_sensor_drsnn, split_time)
X_train_w, X_test_w, y_train_w, y_test_w = test_train_split(df_sensor_window, split_time)

# Time series split of the sampling datasets
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_drsnn)):
    X_train_fold_drsnn, X_val_fold_drsnn = X_train_drsnn.iloc[train_idx], X_train_drsnn.iloc[val_idx]
    y_train_fold_drsnn, y_val_fold_drsnn = y_train_drsnn.iloc[train_idx], y_train_drsnn.iloc[val_idx]

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_w)):
    X_train_fold_w, X_val_fold_w = X_train_w.iloc[train_idx], X_train_w.iloc[val_idx]
    y_train_fold_w, y_val_fold_w = y_train_w.iloc[train_idx], y_train_w.iloc[val_idx]

In [76]:
X_train_drsnn_n, X_test_drsnn_n, y_train_drsnn_n, y_test_drsnn_n = test_train_split(df_sensor_drsnn_norm, split_time)
X_train_w_n, X_test_w_n, y_train_w_n, y_test_w_n = test_train_split(df_sensor_window_norm, split_time)

# Time series split of the sampling datasets
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_drsnn_n)):
    X_train_fold_drsnn_n, X_val_fold_drsnn_n = X_train_drsnn_n.iloc[train_idx], X_train_drsnn_n.iloc[val_idx]
    y_train_fold_drsnn_n, y_val_fold_drsnn_n = y_train_drsnn_n.iloc[train_idx], y_train_drsnn_n.iloc[val_idx]

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_w_n)):
    X_train_fold_w_n, X_val_fold_w_n = X_train_w_n.iloc[train_idx], X_train_w_n.iloc[val_idx]
    y_train_fold_w_n, y_val_fold_w_n = y_train_w_n.iloc[train_idx], y_train_w_n.iloc[val_idx]

In [77]:
gbc = GradientBoostingClassifier(
    n_estimators=30,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    min_samples_leaf=5,
    random_state=42
)
gbc.fit(X_train_fold_drsnn_n,y_train_fold_drsnn_n)
y_pred_proba_n = gbc.predict(X_test_n)

threshold = 0.5
y_pred_n = (y_pred_proba_n >= threshold).astype(int)

In [78]:
gbc = GradientBoostingClassifier(
    n_estimators=30,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    min_samples_leaf=5,
    random_state=42
)
gbc.fit(X_train_fold_w_n,y_train_fold_w_n)
y_pred_proba_w_n = gbc.predict(X_test_n)

threshold = 0.5
y_pred_n = (y_pred_proba_w_n >= threshold).astype(int)

In [79]:
gbc = GradientBoostingClassifier(
    n_estimators=30,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    min_samples_leaf=5,
    random_state=42
)
gbc.fit(X_train_fold_w,y_train_fold_w)
y_pred_proba_w = gbc.predict(X_test_nn)

threshold = 0.5
y_pred = (y_pred_proba_w >= threshold).astype(int)

In [81]:
gbc = GradientBoostingClassifier(
    n_estimators=30,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    min_samples_leaf=5,
    random_state=42
)
gbc.fit(X_train_fold_drsnn,y_train_fold_drsnn)
y_pred_proba_drsnn = gbc.predict(X_test_nn)

threshold = 0.5
y_pred = (y_pred_proba_drsnn >= threshold).astype(int)

In [82]:
def evaluate_model_vs_random(model, X_test, y_test,sampling="name"):
    print(sampling)
    # 1) Predict
    y_pred_proba = model.predict_proba(X_test)[:,1]
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    # 2) Average Precision vs. Random
    ap_model = average_precision_score(y_test, y_pred_proba)
    p = (y_test==1).mean()  # minority proportion
    ap_random = p
    
    print(f"Model AP = {ap_model:.4f}, Random AP = {ap_random:.4f}")
    if ap_model > ap_random:
        print("Better than random (AP).")
    else:
        print("Not better than random (AP).")
    
    # 3) Accuracy vs random
    acc_model = accuracy_score(y_test, y_pred)
    acc_random = max(p, 1-p)
    print(f"Model accuracy: {acc_model:.4f}, Random accuracy: {acc_random:.4f}")
    
    # 4) Binomial test
    from scipy import stats
    model_correct = (y_test == y_pred).sum()
    n_test = len(y_test)
    # Probability of correct if random
    p_value_binom = 1 - stats.binom.cdf(model_correct-1, n_test, acc_random)
    print(f"Binomial test p-value: {p_value_binom:.5f}")
    if p_value_binom < 0.05:
        print("Significantly above random (p<0.05).")
    else:
        print("Not significantly above random (p>=0.05).")


In [83]:
model = GradientBoostingClassifier(random_state=42)
model.fit(X_train_fold_drsnn_n, y_train_fold_drsnn_n)
evaluate_model_vs_random(model, X_test_n, y_test_n,sampling="DRSNN")

DRSNN
Model AP = 0.0003, Random AP = 0.0000
Better than random (AP).
Model accuracy: 0.9801, Random accuracy: 1.0000
Binomial test p-value: 1.00000
Not significantly above random (p>=0.05).


In [84]:
model = GradientBoostingClassifier(random_state=42)
model.fit(X_train_fold_w_n, y_train_fold_w_n)
evaluate_model_vs_random(model, X_test_n, y_test_n,sampling="Window")

Window
Model AP = 0.0005, Random AP = 0.0000
Better than random (AP).
Model accuracy: 0.9804, Random accuracy: 1.0000
Binomial test p-value: 1.00000
Not significantly above random (p>=0.05).


In [85]:
model = GradientBoostingClassifier(random_state=42)
model.fit(X_train_fold_drsnn, y_train_fold_drsnn)
evaluate_model_vs_random(model, X_test_nn, y_test_nn,sampling="DRSNN")

DRSNN
Model AP = 0.9950, Random AP = 0.0205
Better than random (AP).
Model accuracy: 0.9991, Random accuracy: 0.9795
Binomial test p-value: 0.00000
Significantly above random (p<0.05).


In [86]:
model = GradientBoostingClassifier(random_state=42)
model.fit(X_train_fold_w, y_train_fold_w)
evaluate_model_vs_random(model, X_test_nn, y_test_nn,sampling="Window")

Window
Model AP = 0.9960, Random AP = 0.0205
Better than random (AP).
Model accuracy: 0.9991, Random accuracy: 0.9795
Binomial test p-value: 0.00000
Significantly above random (p<0.05).
